<a href="https://colab.research.google.com/github/ARehman007-max/practic-projects/blob/main/tracking_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# One-click install everything
!pip install ultralytics deep-sort-realtime opencv-python-headless numpy --quiet

In [ ]:
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import cv2
import numpy as np
from google.colab import files
from IPython.display import HTML, display
from collections import defaultdict
import pandas as pd
import random, os, warnings
warnings.filterwarnings('ignore')

# GPU check
import torch
print(f"🚀 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"⚡ Using: {torch.cuda.get_device_name(0)}")

# Best model for quality
print("📥 Loading AI Model...")
model = YOLO('yolov8s.pt')  # 's' = small but accurate
print("✅ Ready!")

🚀 GPU Available: False
📥 Loading AI Model...
✅ Ready!


In [ ]:
print("📁 UPLOAD YOUR VIDEO:")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"✅ {video_path} loaded!")

📁 UPLOAD YOUR VIDEO:


Saving people.mp4 to people (1).mp4
✅ people (1).mp4 loaded!


In [ ]:
# ========== ULTRA FAST TRACKER SETTINGS ==========
tracker = DeepSort(
    max_age=20,          # Track lost objects for 20 frames only
    n_init=2,            # Confirm track fast (2 frames)
    max_cosine_distance=0.4,  # Faster matching
    nn_budget=50         # Limit memory for speed
)

colors = {}
track_history = defaultdict(lambda: [])

def get_color(track_id):
    if track_id not in colors:
        colors[track_id] = (
            random.randint(50, 255),
            random.randint(50, 255),
            random.randint(50, 255)
        )
    return colors[track_id]

# ========== OPEN VIDEO ==========
cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Output video - H264 for best quality
fourcc = cv2.VideoWriter_fourcc(*'avc1')
out = cv2.VideoWriter('output.mp4', fourcc, fps, (width, height))

all_data = []
frame_count = 0

print(f"🎬 Processing {total_frames} frames...")
print("="*50)

# ========== PROCESS EACH FRAME ==========
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # FAST DETECTION (reduce size for speed)
    results = model(frame, conf=0.3, verbose=False)[0]  # verbose=False = CLEAN OUTPUT

    # Prepare detections
    detections = []
    frame_confidences = []

    for r in results.boxes.data.tolist():
        x1, y1, x2, y2, confidence, class_id = r
        class_name = results.names[int(class_id)]

        # Only track important objects
        if class_name in ['person', 'car', 'truck', 'motorcycle', 'bus', 'bicycle', 'dog', 'cat', 'bird']:
            detections.append(([int(x1), int(y1), int(x2-x1), int(y2-y1)], confidence, class_name))
            frame_confidences.append(confidence)

    # UPDATE TRACKER
    tracks = tracker.update_tracks(detections, frame=frame)

    # ========== DRAW EVERYTHING ==========
    active_tracks = []

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        ltrb = track.to_ltrb()
        x1, y1, x2, y2 = int(ltrb[0]), int(ltrb[1]), int(ltrb[2]), int(ltrb[3])
        det_class = track.get_det_class()
        color = get_color(track_id)

        # Find confidence
        track_conf = 0.85  # Default if not found
        for i, det in enumerate(detections):
            if det[2] == det_class:
                bx, by, bw, bh = det[0]
                if abs(bx-x1) < 30 and abs(by-y1) < 30:
                    track_conf = det[1]
                    break

        active_tracks.append(track_id)

        # ---- DRAW RECTANGLE (Clean Thick Box) ----
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

        # ---- DRAW LABEL WITH PROBABILITY ----
        label = f"{det_class} | {track_conf:.0%}"

        # Text background
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_DUPLEX, 0.7, 2)
        cv2.rectangle(frame, (x1, y1-th-15), (x1+tw+10, y1), color, -1)

        # Text
        cv2.putText(frame, label, (x1+5, y1-8),
                   cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 2)

        # ---- DRAW ID (Small on top-right) ----
        cv2.putText(frame, f"ID:{track_id}", (x2-70, y1+20),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # ---- MOVEMENT TRAIL ----
        cx, cy = (x1+x2)//2, (y1+y2)//2
        track_history[track_id].append((cx, cy))
        if len(track_history[track_id]) > 20:
            track_history[track_id].pop(0)

        if len(track_history[track_id]) > 1:
            pts = np.array(track_history[track_id], np.int32)
            cv2.polylines(frame, [pts], False, color, 2)

        # Center dot
        cv2.circle(frame, (cx, cy), 4, color, -1)

        # Save data
        all_data.append([frame_count, track_id, det_class, track_conf, cx, cy])

    # ========== TOP BAR: COUNT & PROBABILITY ==========
    # Semi-transparent black bar
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (width, 75), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)

    # Object Count - BIG
    obj_count = len(active_tracks)
    cv2.putText(frame, f"OBJECTS: {obj_count}", (15, 35),
               cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

    # Average Confidence
    if frame_confidences:
        avg_conf = np.mean(frame_confidences)
        conf_color = (0, 255, 0) if avg_conf > 0.7 else (0, 255, 255) if avg_conf > 0.5 else (0, 0, 255)
        cv2.putText(frame, f"CONFIDENCE: {avg_conf:.1%}", (15, 65),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, conf_color, 2)

    # Frame counter
    cv2.putText(frame, f"Frame: {frame_count}/{total_frames}", (width-280, 65),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # ========== SAVE FRAME ==========
    out.write(frame)

    # Progress update (Har 50 frames mein)
    if frame_count % 50 == 0:
        print(f"✅ {frame_count}/{total_frames} | Objects: {obj_count} | Confidence: {avg_conf:.1%}")

# ========== FINALIZE ==========
cap.release()
out.release()

# Save CSV
if all_data:
    df = pd.DataFrame(all_data, columns=['Frame', 'Track_ID', 'Class', 'Confidence', 'X', 'Y'])
    df.to_csv('tracking_data.csv', index=False)

print("="*50)
print(f"🎉 COMPLETE! Total objects tracked: {len(set(t['track_id'] for t in tracks if hasattr(t, 'track_id')))}")

🎬 Processing 341 frames...
✅ 50/341 | Objects: 37 | Confidence: 55.7%
✅ 100/341 | Objects: 41 | Confidence: 60.2%
✅ 150/341 | Objects: 32 | Confidence: 60.8%
✅ 200/341 | Objects: 29 | Confidence: 59.7%
✅ 250/341 | Objects: 34 | Confidence: 64.6%
✅ 300/341 | Objects: 36 | Confidence: 57.1%


TypeError: 'Track' object is not subscriptable